# Capítulo 5 — Simuladores Interactivos
## Taxias y Sistemas de Retroalimentación

Este notebook contiene tres simuladores para explorar los conceptos del capítulo.
Ejecuta primero la celda de instalación, luego cada simulador de forma independiente.

---

In [ ]:
# Celda 0 — Instalación (ejecutar primero, una sola vez)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ipywidgets import interact, FloatSlider, IntSlider, fixed
print('Listo.')

---
## Simulador 1 — Fototaxia: el efecto de la ganancia

Un organismo parte de un ángulo inicial θ₀ respecto a la fuente de luz.
La función de control es **ω = k · sin(θ)**: el giro es proporcional al ángulo.

**Explora:**
- ¿Qué pasa con la trayectoria cuando la ganancia `k` es muy alta? ¿Muy baja?
- ¿El punto de equilibrio cambia con `k`, o solo la velocidad de convergencia?
- ¿Qué diferencia hay entre partir de θ₀ = 30° y θ₀ = 150°?

In [ ]:
def sim1_fototaxia(k=1.0, theta0=75.0):
    """
    Simulador 1: Fototaxia básica.
    Muestra dos paneles: ángulo vs. tiempo, y trayectoria 2D.
    """
    dt = 0.05
    T = 20.0
    theta = np.radians(theta0)
    
    # --- Simulación ---
    thetas = [np.degrees(theta)]
    times  = [0.0]
    # Posición 2D: organismo empieza en (0,0), fuente en (0, 10)
    px, py = 0.0, 0.0          # posición organismo
    vx = np.sin(theta)         # dirección inicial de movimiento
    vy = np.cos(theta)
    xs, ys = [px], [py]
    speed = 0.3

    t = 0.0
    while t < T:
        error = np.sin(theta)
        omega = k * error
        theta -= omega * dt
        # Mover organismo en la dirección actual
        px += speed * np.sin(theta) * dt
        py += speed * np.cos(theta) * dt
        t += dt
        thetas.append(np.degrees(theta))
        times.append(t)
        xs.append(px)
        ys.append(py)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Panel izquierdo: ángulo vs. tiempo
    ax1.plot(times, thetas, color='steelblue', lw=2)
    ax1.axhline(0, color='tomato', linestyle='--', lw=1.5, label='Equilibrio (θ = 0°)')
    ax1.set_xlabel('Tiempo')
    ax1.set_ylabel('Ángulo θ (grados)')
    ax1.set_title(f'Convergencia  |  k = {k:.1f},  θ₀ = {theta0:.0f}°')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-20, max(theta0 + 10, 20))

    # Panel derecho: trayectoria 2D
    ax2.plot(xs, ys, color='steelblue', lw=1.5, label='Trayectoria')
    ax2.plot(xs[0], ys[0], 'o', color='green', ms=8, label='Inicio')
    ax2.plot(0, 10, '*', color='gold', ms=14, markeredgecolor='black', label='Fuente de luz')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    ax2.set_title('Trayectoria 2D')
    ax2.legend()
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

interact(
    sim1_fototaxia,
    k      = FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1,
                         description='Ganancia k',
                         style={'description_width': '120px'},
                         layout={'width': '450px'}),
    theta0 = FloatSlider(value=75.0, min=5.0, max=170.0, step=5.0,
                         description='Ángulo inicial θ₀',
                         style={'description_width': '120px'},
                         layout={'width': '450px'})
);

---
## Simulador 2 — El efecto de la demora

En un sistema de retroalimentación real, siempre hay una demora entre que el sensor detecta el error y que el efector ejecuta la corrección.
Aquí `demora` es el número de pasos de tiempo que tarda la corrección en aplicarse.

**Explora:**
- Con demora = 0, ¿cómo converge el sistema?
- ¿A partir de qué valor de demora empiezan a aparecer oscilaciones?
- Si aumentas `k` y la demora al mismo tiempo, ¿qué pasa? ¿Por qué interactúan?
- ¿Puedes hacer que el sistema sea inestable (oscilaciones que crecen en lugar de amortiguar)?

In [ ]:
from collections import deque

def sim2_demora(k=1.0, theta0=75.0, demora=0):
    """
    Simulador 2: Efecto de demora en la retroalimentación.
    """
    dt = 0.05
    T = 30.0
    theta = np.radians(theta0)
    buffer = deque([0.0] * max(demora, 1), maxlen=max(demora, 1))

    thetas = [np.degrees(theta)]
    times  = [0.0]
    t = 0.0

    while t < T:
        error = np.sin(theta)
        if demora == 0:
            omega = k * error
        else:
            omega = buffer.popleft()
            buffer.append(k * error)
        theta -= omega * dt
        t += dt
        thetas.append(np.degrees(theta))
        times.append(t)

    # Color según comportamiento
    max_abs = max(abs(v) for v in thetas[len(thetas)//2:])
    if max_abs < 2:
        color = 'steelblue'
        estado = 'Convergencia estable'
    elif max_abs < 20:
        color = 'darkorange'
        estado = 'Oscilaciones amortiguadas'
    else:
        color = 'tomato'
        estado = '¡Sistema inestable!'

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(times, thetas, color=color, lw=2)
    ax.axhline(0, color='gray', linestyle='--', lw=1, alpha=0.7)
    ax.set_xlabel('Tiempo')
    ax.set_ylabel('Ángulo θ (grados)')
    ax.set_title(f'{estado}  |  k = {k:.1f},  demora = {demora} pasos,  θ₀ = {theta0:.0f}°')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-180, 180)
    plt.tight_layout()
    plt.show()

interact(
    sim2_demora,
    k      = FloatSlider(value=1.0, min=0.1, max=4.0, step=0.1,
                         description='Ganancia k',
                         style={'description_width': '120px'},
                         layout={'width': '450px'}),
    theta0 = FloatSlider(value=75.0, min=5.0, max=170.0, step=5.0,
                         description='Ángulo inicial θ₀',
                         style={'description_width': '120px'},
                         layout={'width': '450px'}),
    demora = IntSlider(value=0, min=0, max=30, step=1,
                       description='Demora (pasos)',
                       style={'description_width': '120px'},
                       layout={'width': '450px'})
);

---
## Simulador 3 — Conductor reactivo vs. anticipatorio

Un conductor viaja 500 km. Las gasolineras están distribuidas de forma irregular.
El **conductor reactivo** carga gasolina solo cuando el tanque baja del umbral.
El **conductor anticipatorio** carga siempre que haya gasolinera disponible
y el tramo siguiente sea más largo que su autonomía restante.

**Explora:**
- Con la distribución por defecto, ¿cuál conductor llega?
- Mueve el slider de `umbral_reactivo` hacia arriba: ¿en qué punto el reactivo también llega?
- Prueba una distribución con tramos muy largos (modifica directamente las posiciones en la celda).
- ¿Qué tan alto tiene que estar el umbral reactivo para que sea equivalente al anticipatorio?
  ¿Qué costo tiene esa estrategia?

In [ ]:
# Puedes cambiar estas posiciones para explorar distintas rutas
GASOLINERAS_KM = [40, 90, 250, 280, 450]  # km donde hay gasolinera
AUTONOMIA_KM   = 100                       # tanque lleno = 100 km
RUTA_KM        = 500

def sim3_conductor(umbral_reactivo=0.15):
    """
    Simulador 3: Reactivo vs. Anticipatorio.
    umbral_reactivo: fracción del tanque (0-1) que dispara la carga en el conductor reactivo.
    """
    def run(estrategia):
        pos = 0
        tanque = 1.0
        cargas = []
        tanques = [1.0]
        posiciones = [0]
        varado = False
        paso = 5  # km por paso

        while pos < RUTA_KM:
            en_gasolinera = any(abs(pos - g) < paso/2 for g in GASOLINERAS_KM)

            if en_gasolinera:
                if estrategia == 'reactivo':
                    if tanque < umbral_reactivo:
                        tanque = 1.0
                        cargas.append(pos)
                else:  # anticipatorio
                    km_prox = min(
                        (g - pos for g in GASOLINERAS_KM if g > pos),
                        default=RUTA_KM - pos
                    )
                    necesito = km_prox / AUTONOMIA_KM + 0.05
                    if tanque < min(necesito, 1.0):
                        tanque = 1.0
                        cargas.append(pos)

            tanque -= paso / AUTONOMIA_KM
            pos += paso

            if tanque <= 0:
                varado = True
                tanque = 0
                tanques.append(tanque)
                posiciones.append(pos)
                break

            tanques.append(tanque)
            posiciones.append(pos)

        return posiciones, tanques, cargas, varado

    pos_r, tank_r, cargas_r, varado_r = run('reactivo')
    pos_a, tank_a, cargas_a, varado_a = run('anticipatorio')

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    # Panel superior: nivel de tanque
    ax1.plot(pos_r, tank_r, color='tomato',    lw=2, label=f'Reactivo (umbral={umbral_reactivo:.0%})')
    ax1.plot(pos_a, tank_a, color='steelblue', lw=2, label='Anticipatorio')
    for g in GASOLINERAS_KM:
        ax1.axvline(g, color='green', lw=1, alpha=0.4, linestyle=':')
    ax1.axhline(umbral_reactivo, color='tomato', lw=1, linestyle='--', alpha=0.6)
    ax1.set_ylabel('Nivel del tanque (fracción)')
    ax1.set_ylim(-0.05, 1.1)
    ax1.legend()
    ax1.grid(True, alpha=0.2)
    ax1.set_title('Nivel de gasolina a lo largo del viaje')

    # Marcar cargas
    for c in cargas_r:
        idx = min(range(len(pos_r)), key=lambda i: abs(pos_r[i]-c))
        ax1.plot(c, tank_r[idx], 'v', color='tomato', ms=8)
    for c in cargas_a:
        idx = min(range(len(pos_a)), key=lambda i: abs(pos_a[i]-c))
        ax1.plot(c, tank_a[idx], 'v', color='steelblue', ms=8)

    # Marcar si se varó
    if varado_r:
        ax1.axvline(pos_r[-1], color='tomato', lw=2, linestyle='-')
        ax1.text(pos_r[-1]+2, 0.5, '¡Varado!', color='tomato', fontsize=10)

    # Panel inferior: distancia recorrida como barra
    resultado_r = f'Reactivo: km {pos_r[-1]:.0f}{" ← VARADO" if varado_r else " ✓"}'
    resultado_a = f'Anticipatorio: km {pos_a[-1]:.0f}{" ← VARADO" if varado_a else " ✓"}'
    ax2.barh(['Anticipatorio', 'Reactivo'],
             [pos_a[-1], pos_r[-1]],
             color=['steelblue', 'tomato'], alpha=0.8)
    ax2.axvline(RUTA_KM, color='black', lw=1.5, linestyle='--', label=f'Meta: {RUTA_KM} km')
    for g in GASOLINERAS_KM:
        ax2.axvline(g, color='green', lw=1, alpha=0.4, linestyle=':')
    ax2.set_xlabel('Kilómetros recorridos')
    ax2.set_xlim(0, RUTA_KM + 20)
    ax2.legend(loc='lower right')
    ax2.grid(True, alpha=0.2, axis='x')
    ax2.set_title(f'{resultado_r}     |     {resultado_a}')

    gasolineras_patch = mpatches.Patch(color='green', alpha=0.4, label='Gasolineras')
    ax2.legend(handles=[gasolineras_patch,
                         mpatches.Patch(color='black', label=f'Meta: {RUTA_KM} km')],
               loc='lower right')

    plt.tight_layout()
    plt.show()
    print(f'Gasolineras en km: {GASOLINERAS_KM}')
    print(f'Cargas reactivo:      {cargas_r}')
    print(f'Cargas anticipatorio: {cargas_a}')

interact(
    sim3_conductor,
    umbral_reactivo = FloatSlider(value=0.15, min=0.05, max=0.95, step=0.05,
                                  description='Umbral reactivo',
                                  style={'description_width': '130px'},
                                  layout={'width': '450px'},
                                  readout_format='.0%')
);